In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Optimization Solver: Funkcja Qiskit od Q-CTRL Fire Opal
*Zobacz [dokumentację API](https://docs.quantum.ibm.com/api/functions/q-ctrl-optimization-solver)*

> **Note:** Funkcje Qiskit to funkcja eksperymentalna dostępna wyłącznie dla użytkowników planów IBM Quantum&reg; Premium Plan, Flex Plan oraz On-Prem (przez IBM Quantum Platform API). Są one w stanie podglądu (preview) i mogą ulec zmianie.


<Accordion>
<AccordionItem title="Wersje pakietów">

Kod na tej stronie został opracowany przy użyciu następujących wymagań.
Zalecamy używanie tych lub nowszych wersji.

```
qiskit-ibm-runtime~=0.46.1
sympy~=1.14.0
```
</AccordionItem>
</Accordion>
## Przegląd
Dzięki Fire Opal Optimization Solver możesz rozwiązywać problemy optymalizacyjne w skali użytkowej na sprzęcie kwantowym bez konieczności posiadania wiedzy z zakresu informatyki kwantowej. Po prostu podaj definicję problemu na wysokim poziomie, a Solver zajmie się resztą. Cały przepływ pracy jest świadomy szumów i pod spodem korzysta z [Fire Opal Performance Management](/guides/q-ctrl-performance-management). Solver konsekwentnie dostarcza dokładnych rozwiązań klasycznie trudnych problemów, nawet w pełnej skali urządzenia na największych QPU IBM&reg;.

Solver jest elastyczny i można go używać do rozwiązywania kombinatorycznych problemów optymalizacyjnych zdefiniowanych jako funkcje celu lub dowolne grafy. Problemy nie muszą być mapowane na topologię urządzenia. Można rozwiązywać zarówno problemy nieograniczone, jak i ograniczone, pod warunkiem że ograniczenia można sformułować jako człony kary. Przykłady zawarte w tym przewodniku pokazują, jak rozwiązać nieograniczony i ograniczony problem optymalizacyjny w skali użytkowej przy użyciu różnych typów danych wejściowych Solvera. Pierwszy przykład dotyczy problemu Max-Cut zdefiniowanego na grafie 3-regularnym o 156 węzłach, podczas gdy drugi przykład dotyczy problemu Minimalnego Pokrycia Wierzchołkowego (Minimum Vertex Cover) na 50 węzłach zdefiniowanego przez funkcję kosztu.

Aby uzyskać dostęp do Optimization Solver, [skontaktuj się z Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Opis funkcji
Solver w pełni optymalizuje i automatyzuje cały algorytm – od tłumienia błędów na poziomie sprzętu po wydajne mapowanie problemu i zamkniętą pętlę optymalizacji klasycznej. Za kulisami potok Solvera redukuje błędy na każdym etapie, umożliwiając zwiększoną wydajność niezbędną do sensownego skalowania. Bazowy przepływ pracy jest inspirowany Quantum Approximate Optimization Algorithm (QAOA), który jest hybrydowym algorytmem kwantowo-klasycznym. Szczegółowe podsumowanie pełnego przepływu pracy Optimization Solver znajdziesz w [opublikowanym artykule](https://arxiv.org/abs/2406.01743).

![Wizualizacja przepływu pracy Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Aby rozwiązać ogólny problem za pomocą Optimization Solver:
1. Zdefiniuj swój problem jako funkcję celu, graf lub łańcuch spinowy `SparsePauliOp`.
2. Połącz się z funkcją przez katalog Qiskit Functions Catalog.
3. Uruchom problem za pomocą Solvera i pobierz wyniki.
### Akceptowane formaty problemu
- Reprezentacja wyrażenia wielomianowego funkcji celu. Najlepiej tworzona w Pythonie za pomocą istniejącego obiektu SymPy Poly i formatowana do postaci ciągu znaków przy użyciu [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- Reprezentacja grafowa określonego typu problemu. Graf powinien być tworzony przy użyciu biblioteki networkx w Pythonie, a następnie konwertowany do ciągu znaków za pomocą funkcji networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- Reprezentacja łańcucha spinowego określonego problemu. Łańcuch spinowy powinien być reprezentowany jako obiekt `SparsePauliOp`; więcej szczegółów znajdziesz w [dokumentacji](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp).

### Obsługiwane Backend-y
Uruchom następujący kod, aby zobaczyć listę Backend-ów, które są aktualnie obsługiwane. Jeśli twojego urządzenia nie ma na liście, [skontaktuj się z Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com), aby dodać obsługę.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

service.backends()

[<IBMBackend('ibm_boston')>,
 <IBMBackend('ibm_pittsburgh')>,
 <IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_marrakesh')>,
 <IBMBackend('ibm_kingston')>,
 <IBMBackend('ibm_miami')>]

## Benchmarks

[Published benchmarking results](https://arxiv.org/abs/2406.01743) show that the Solver successfully solves problems with over 120 qubits, even outperforming previously published results on quantum annealing and trapped-ion devices. The following benchmark metrics provide a rough indication of the accuracy and scaling of problem types based on a few examples. Actual metrics may differ based on various problem features, such as the number of terms in the objective function (density) and their locality, number of variables, and polynomial order.

The "Number of qubits" indicated is not a hard limitation but represents rough thresholds where you can expect extremely consistent solution accuracy. Larger problem sizes have been successfully solved, and testing beyond these limits is encouraged.

Arbitrary qubit connectivity is supported across all problem types.

| Problem type    | Number of qubits | Example | Accuracy | Total time (s) | Runtime usage (s) | Number of iterations
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Sparsely-connected quadratic problems  | 156 | 3-regular max-cut| 100%     | 1764     | 293          | 16 |
| Higher-order binary optimization | 156 | Ising spin-glass model | 100%      | 1461     | 272          | 16 |
| Densely-connected quadratic problems | 50 | Fully-connected max-cut| 100%      |  1758    | 268  | 12 |
| Constrained problem with penalty terms | 50 | Weighted Minimum Vertex Cover with 8% edge density | 100%      | 1074     | 215 | 10 |

## Get started

First, authenticate using your [IBM Quantum API key](http://quantum.cloud.ibm.com/). Then, select the Qiskit Function as follows. (This snippet assumes you've already [saved your account](/docs/guides/functions#install-qiskit-functions-catalog-client) to your local environment.)

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

## Benchmarki

[Opublikowane wyniki testów porównawczych](https://arxiv.org/abs/2406.01743) pokazują, że Solver skutecznie rozwiązuje problemy z ponad 120 Qubitami, a nawet przewyższa wcześniej opublikowane wyniki na urządzeniach do wyżarzania kwantowego i pułapkowania jonów. Poniższe metryki benchmarkowe dają przybliżone wskazanie dokładności i skalowalności typów problemów na podstawie kilku przykładów. Rzeczywiste metryki mogą się różnić w zależności od różnych cech problemu, takich jak liczba składników w funkcji celu (gęstość) i ich lokalność, liczba zmiennych oraz rząd wielomianu.

„Liczba Qubitów" podana w tabeli nie jest twardym ograniczeniem, lecz przybliżonym progiem, powyżej którego możesz oczekiwać bardzo spójnej dokładności rozwiązania. Większe rozmiary problemów zostały z powodzeniem rozwiązane i zachęcamy do testowania powyżej tych limitów.

Dowolna łączność Qubitów jest obsługiwana dla wszystkich typów problemów.

| Typ problemu    | Liczba Qubitów | Przykład | Dokładność | Całkowity czas (s) | Użycie środowiska uruchomieniowego (s) | Liczba iteracji
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Rzadko połączone problemy kwadratowe  | 156 | max-cut 3-regularny | 100%     | 1764     | 293          | 16 |
| Optymalizacja binarna wyższego rzędu | 156 | Model szkła spinowego Isinga | 100%      | 1461     | 272          | 16 |
| Gęsto połączone problemy kwadratowe | 50 | max-cut w pełni połączony | 100%      |  1758    | 268  | 12 |
| Problem ograniczony z członami kary | 50 | Ważone Minimalne Pokrycie Wierzchołkowe z gęstością krawędzi 8% | 100%      | 1074     | 215 | 10 |

## Pierwsze kroki

Najpierw uwierzytelnij się przy użyciu swojego [klucza API IBM Quantum](http://quantum.cloud.ibm.com/). Następnie wybierz Funkcję Qiskit w sposób podany poniżej. (Ten fragment zakłada, że masz już [zapisane konto](/guides/functions#install-qiskit-functions-catalog-client) w lokalnym środowisku.)

In [3]:
# %pip install networkx numpy

## Przykład: Optymalizacja nieograniczona
Uruchom problem [maksymalnego cięcia](https://en.wikipedia.org/wiki/Maximum_cut) (max-cut). Poniższy przykład demonstruje możliwości Solvera na problemie max-cut na nieważonym grafie 3-regularnym o 156 węzłach, ale możesz też rozwiązywać problemy na grafach ważonych.
Oprócz `qiskit-ibm-catalog` będziesz również używać następujących pakietów do uruchomienia tego przykładu: `networkx` i `numpy`. Możesz zainstalować te pakiety, odkomentowując poniższą komórkę, jeśli uruchamiasz ten przykład w notatniku z jądrem IPython.

In [4]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [5]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.svg" alt="Output of the previous code cell" />

### 1. Zdefiniuj problem
Możesz uruchomić problem Max-Cut, definiując problem grafowy i określając `problem_type='maxcut'`.

In [6]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

### 2. Run the problem
When using the graph-based input method, specify the problem type.

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [8]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

### 2. Uruchom problem
Gdy używasz metody wejściowej opartej na grafie, określ typ problemu.

In [9]:
# Get job status
print(maxcut_job.status())

QUEUED


### 3. Retrieve the result
Retrieve the optimal cut value from the results dictionary.

<Admonition type="note">
   The mapping of the variables to the bitstring may have changed. The output dictionary contains a `variables_to_bitstring_index_map` sub-dictionary, which helps to verify the ordering.
</Admonition>

In [10]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 210.0


Sprawdź [status](/guides/functions#check-job-status) obciążenia swojej Funkcji Qiskit lub pobierz [wyniki](/guides/functions#retrieve-results) w następujący sposób:

In [11]:
# %pip install numpy networkx sympy

### 1. Define the problem
Define a random MVC problem by generating a graph with randomly weighted nodes.

In [12]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg" alt="Output of the previous code cell" />

### 3. Pobierz wynik
Pobierz optymalną wartość cięcia ze słownika wyników.

> **Note:** Mapowanie zmiennych do bitstring-u mogło ulec zmianie. Słownik wyjściowy zawiera pod-słownik `variables_to_bitstring_index_map`, który pomaga zweryfikować kolejność.

In [13]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

Now every edge in the graph should include at least one end point from the cover, which can be expressed as the inequality:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Any case where an edge is not connected to the vertex of cover must be penalized. This can be represented in the cost function by adding a penalty of the form $P(1-n_i-n_j+n_i n_j)$ where $P$ is a positive penalty constant. Thus, an unconstrained alternative to the constrained inequality for weighted MVC is:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [14]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

Możesz zweryfikować dokładność wyniku, rozwiązując problem klasycznie za pomocą solverów open-source, takich jak [PuLP](https://coin-or.github.io/pulp/), jeśli graf nie jest gęsto połączony. W przypadku problemów o dużej gęstości do walidacji rozwiązania mogą być wymagane zaawansowane solwery klasyczne.
## Przykład: Optymalizacja ograniczona
Poprzedni przykład max-cut to popularny problem kwadratowej binarnej optymalizacji bez ograniczeń. Optimization Solver Q-CTRL można stosować do różnych typów problemów, w tym optymalizacji ograniczonej. Możesz rozwiązywać dowolne typy problemów, podając definicję problemu reprezentowaną jako wielomian, w którym ograniczenia są modelowane jako człony kary.

Poniższy przykład pokazuje, jak skonstruować funkcję kosztu dla ograniczonego problemu optymalizacyjnego, [minimalnego pokrycia wierzchołkowego](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
Oprócz pakietów `qiskit-ibm-catalog` i `qiskit` będziesz również używać następujących pakietów do uruchomienia tego przykładu: `numpy`, `networkx` i `sympy`. Możesz zainstalować te pakiety, odkomentowując poniższą komórkę, jeśli uruchamiasz ten przykład w notatniku z jądrem IPython.

In [15]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

### 1. Zdefiniuj problem
Zdefiniuj losowy problem MVC, generując graf z losowo ważonymi węzłami.

In [16]:
print(mvc_job.status())

QUEUED


![Wynik poprzedniej komórki kodu](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg)

Standardowy model optymalizacyjny dla ważonego MVC można sformułować w następujący sposób. Po pierwsze, należy dodać karę za każdy przypadek, gdy krawędź nie jest połączona z wierzchołkiem w podzbiorze. Dlatego niech $n_i = 1$, jeśli wierzchołek $i$ należy do pokrycia (tj. do podzbioru) i $n_i = 0$ w przeciwnym razie. Po drugie, celem jest minimalizacja łącznej liczby wierzchołków w podzbiorze, co można wyrazić następującą funkcją:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [17]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


Teraz każda krawędź w grafie powinna zawierać co najmniej jeden punkt końcowy z pokrycia, co można wyrazić jako nierówność:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Każdy przypadek, gdy krawędź nie jest połączona z wierzchołkiem pokrycia, musi być ukarany. Można to przedstawić w funkcji kosztu, dodając karę postaci $P(1-n_i-n_j+n_i n_j)$, gdzie $P$ jest dodatnią stałą kary. Zatem nieograniczona alternatywa dla ograniczonej nierówności dla ważonego MVC to:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$